In [1]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

# Load your code.env file
env_file = r"C:\Users\Abhijeet\code.env"

with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

print(f" Loaded from: {env_file}")

ok_openai = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI : {' Loaded' if ok_openai else ' Not found'}")

 Loaded from: C:\Users\Abhijeet\code.env
OpenAI :  Loaded


In [3]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"],
               capture_output=True)

# Load from code.env
env_file = r"C:\Users\Abhijeet\code.env"
with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

# Anthropic not needed — set dummy so scraper skips quietly
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = "not-used"

ok = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI key : {' Ready' if ok else ' Not found — check code.env'}")
if ok:
    print(" Ready to run scraper")

OpenAI key :  Ready
 Ready to run scraper


In [5]:
import asyncio
import re
import sys
import json
import urllib.request
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


# ── SENTENCE-SAFE TRUNCATION ──────────────────────────────────────────────────

def clean_to_sentences(text: str, max_sentences: int = 2) -> str:
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    good = [
        s.strip() for s in sentences
        if len(s.strip().split()) >= 6 and re.search(r'[.!?]$', s.strip())
    ]
    return " ".join(good[:max_sentences])


# ── API KEY ───────────────────────────────────────────────────────────────────

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def _get_openai_key() -> str:
    return os.environ.get("OPENAI_API_KEY", "")

def _check_key(key: str, name: str) -> bool:
    if not key or key.startswith("YOUR_") or len(key) < 20:
        print(f"  ⚠ {name} not set — add it to your .env file")
        return False
    return True


# ── HELPERS ───────────────────────────────────────────────────────────────────

async def highlight_and_click(page, selector_or_locator, description="Action"):
    try:
        target = (
            page.locator(selector_or_locator).first
            if isinstance(selector_or_locator, str)
            else selector_or_locator
        )
        if await target.is_visible(timeout=2000):
            await target.evaluate(
                "el => { el.style.border = '4px solid #00FF00';"
                " el.style.backgroundColor = 'rgba(0,255,0,0.2)'; }"
            )
            await target.click()
            return True
    except Exception:
        pass
    return False


async def handle_cookies_automatically(page):
    for selector in [
        "text=Accept All", "text=Accept",
        "button:has-text('Accept')", "button:has-text('OK')",
        "button:has-text('I Agree')", "button:has-text('Close')",
        "button:has-text('Got it')", "button:has-text('Agree')",
    ]:
        if await highlight_and_click(page, selector, "Cookie Banner"):
            break


async def smart_goto(page, url, timeout=90000):
    try:
        await page.goto(url, wait_until="networkidle", timeout=timeout)
        return True
    except Exception as e:
        print(f"  ⚠ networkidle failed ({e.__class__.__name__}), retrying…")
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
        await asyncio.sleep(6)
        try:
            await page.wait_for_function(
                "() => document.body && document.body.innerText.trim().length > 200",
                timeout=15000)
        except Exception:
            pass
        return True
    except Exception as e:
        try:
            await page.goto(url, wait_until="commit", timeout=timeout)
            await asyncio.sleep(8)
            return True
        except Exception:
            pass
        print(f"  ✗ Failed: {e}")
        return False


# ── KNOWN NAMES MAP — city locked, always wins over page title ────────────────
# Format: domain_fragment: (correct_name, correct_city)

KNOWN_NAMES_MIA = {
    "ransomeverglades.org":  ("Ransom Everglades School",                "Coconut Grove, FL"),
    "gulliverschools.org":   ("Gulliver Schools",                        "Pinecrest, FL"),
    "gulliverprep.org":      ("Gulliver Prep",                           "Pinecrest, FL"),
    "carrollton.org":        ("Carrollton School of the Sacred Heart",   "Miami, FL"),
    "palmertrinity.org":     ("Palmer Trinity School",                   "Palmetto Bay, FL"),
    "miamicountryday.org":   ("Miami Country Day School",                "Miami, FL"),
    "imgacademy.com":        ("IMG Academy",                             "Bradenton, FL"),
    "polotraining.org":      ("Polo Training Foundation",                "Wellington, FL"),
    "crandonparktennis.com": ("Crandon Park Tennis Center",              "Key Biscayne, FL"),
    "princetonreview.com":   ("The Princeton Review",                    "Miami, FL"),
    "baptisteyoga.com":      ("Baron Baptiste Yoga",                     "Miami, FL"),
    "miamimusicproject.org": ("Miami Music Project",                     "Miami, FL"),
    "nwsa.mdc.edu":          ("New World School of the Arts",            "Miami, FL"),
    "miamibeachfl.gov":      ("Miami Beach Tennis Center",               "Miami Beach, FL"),
}

# Verified factual data — never overwritten by scraping
# Ages = true student age range (not grade numbers)
KNOWN_DATA_MIA = {
    # Ransom Everglades: grades 6-12 = ages 11-18; ~1,200 students; founded 1903
    "ransomeverglades.org": {
        "Ages": "11–18", "Students": "1200", "Founded": "1903",
        "Type": "Independent Day School",
    },
    # Gulliver Schools: PreK-12 = ages 4-18; ~2,200 students; founded 1926
    "gulliverschools.org": {
        "Ages": "4–18", "Students": "2200", "Founded": "1926",
        "Type": "Independent K-12 School",
    },
    "gulliverprep.org": {
        "Ages": "4–18", "Students": "2200", "Founded": "1926",
        "Type": "Independent K-12 School",
    },
    # Carrollton: grades K-12 (girls) = ages 5-18; ~850 students; founded 1961
    "carrollton.org": {
        "Ages": "5–18", "Students": "850", "Founded": "1961",
        "Type": "Independent All-Girls School",
    },
    # Palmer Trinity: grades 6-12 = ages 11-18; ~805 students; founded 1992
    "palmertrinity.org": {
        "Ages": "11–18", "Students": "805", "Founded": "1992",
        "Type": "Independent Day School",
    },
    # Miami Country Day: PreK-12 = ages 4-18; ~1,380 students; founded 1938
    "miamicountryday.org": {
        "Ages": "4–18", "Students": "1380", "Founded": "1938",
        "Type": "Independent K-12 School",
    },
    # IMG Academy: boarding school grades 6-PG = ages 11-19; ~1,300 students; founded 1978
    "imgacademy.com": {
        "Ages": "11–19", "Students": "1300", "Founded": "1978",
        "Type": "Sports Performance Academy & Boarding School",
    },
    # Polo Training Foundation: youth polo program ages 8-18; Wellington FL
    "polotraining.org": {
        "Ages": "8–18", "Students": "N/A", "Founded": "1967",
        "Type": "Polo Training Foundation",
    },
    # Crandon Park Tennis: public facility, all ages; Key Biscayne FL; 1949
    "crandonparktennis.com": {
        "Ages": "All ages", "Students": "N/A", "Founded": "1949",
        "Type": "Public Tennis Centre",
    },
    # Princeton Review: test prep ages 13-18
    "princetonreview.com": {
        "Ages": "13–18", "Students": "N/A", "Founded": "1981",
        "Type": "Academic Test Prep & Tutoring",
    },
    # Baron Baptiste Yoga: online yoga training, adults 18+
    "baptisteyoga.com": {
        "Ages": "Adults 18+", "Students": "N/A", "Founded": "1992",
        "Type": "Yoga Training & Education Platform",
    },
    # Miami Music Project: after-school music; ages 5-18; founded 2008
    "miamimusicproject.org": {
        "Ages": "5–18", "Students": "850", "Founded": "2008",
        "Type": "Youth Music Programme",
    },
    # NWSA: high school + college arts; ages 14-22; founded 1984
    "nwsa.mdc.edu": {
        "Ages": "14–22", "Students": "500", "Founded": "1984",
        "Type": "Performing & Visual Arts School",
    },
    # Miami Beach Tennis Center: public facility all ages
    "miamibeachfl.gov": {
        "Ages": "All ages", "Students": "N/A", "Founded": "1964",
        "Type": "Public Tennis Centre",
    },
}


# ── SHORT KEYWORDS MAP for WP blocks ──────────────────────────────────────────

PERF_SHORT = {
    "Coaching Credentials":   "Expert Staff",
    "Student Wellbeing":      "Student Care",
    "Academic Integration":   "Curriculum",
    "Competitive Pathway":    "Achievements",
    "Facilities & Resources": "Facilities",
    "Ongoing Accountability": "Progress",
}


# ── GARBAGE SITE DETECTION ────────────────────────────────────────────────────

GARBAGE_SIGNALS = [
    "hugedomains", "domain for sale", "buy this domain", "sedoparking",
    "this domain is for sale", "godaddy", "namecheap parking",
    "page not found", "account suspended", "coming soon",
    "under construction", "website coming soon",
]

def is_garbage_site(name: str, body_text: str) -> bool:
    combined = (name + " " + body_text[:500]).lower()
    return any(sig in combined for sig in GARBAGE_SIGNALS)


# ── JUNK FILTER ───────────────────────────────────────────────────────────────

JUNK_PATTERNS = [
    # Generic UI / navigation
    r"the site you are about to visit", r"page you requested no longer exists",
    r"managed independently", r"beginning of dialog window", r"escape will cancel",
    r"enquire now$", r"apply now$", r"watch the video", r"learn more$",
    r"find out more$", r"read more$", r"click here",
    r"cookie", r"privacy policy", r"gdpr",
    r"please fill out the form", r"fill the registration form",
    r"mailing list.*keep you up to date", r"joining our mailing list",
    # Legal boilerplate
    r'"school" means the school', r"student.*means the child",
    r"parents.*guardians.*applying",
    r"to progress your application we require",
    r"please see below the list of documents",
    r"documents that are mandatory to upload",
    r"our system is limited to uploading",
    r"scans of multi-page documents",
    r"tourist visa.*online",
    r"carrollton does not discriminate in admissions.*on the basis",
    # Miami-specific junk patterns from the log
    r"img academy will provide sheets",
    r"pillow and blanket",
    r"camper did not receive sheets",
    r"your suggestion box",
    r"community satisfaction survey",
    r"document\.getelementbyid", r"ak_js",
    r"accessibility statement", r"skip to main content",
    r"city commission", r"city manager", r"commissioner",
    r"officers of the board occupy the leadership",
    r"remaining trustees outside of the officers",
    r"it is both a privilege and a sacred duty",
    r"sacred heart school.*witness god's grace",
    r"heather is an alumna of carrollton",
    r"harvard university.*barry university",
    r"we warmly welcome you to the 2025",
    r"this is an inspiring and transformative time",
    r"parents are responsible for requesting.*transcripts",
    r"these must be submitted electronically by the student",
    r"if you have questions regarding the admission process.*email admissions",
    r"rental.*facilities.*based on availability",
    r"please connect with your service specialist",
    r"number of campers per room is subject to change",
    r"esl course with weekly themes",
    r"inspiring people.*extreme sports.*endangered planet",
    r"custom-curated experience.*movies poolside",
    r"cultural dining experiences.*trips to local beaches",
    r"©\d{4} tpr education",
    r"the princeton review is not affiliated with princeton university",
    r"sat is a registered trademark of the college board",
    r"if your starting score is below 1250.*refund",
    r"if your starting score is 1250 or higher",
    r"we surveyed 150 institutions to rank",
    r"best classroom experience.*best college radio station",
    r"study when and where it works best.*by the pool",
    r"from best classroom experience",
    r"dog parks.*fitness.*sports.*playgrounds",
    r"times for the public comment period.*stanley sutnick",
    r"dr\. stanley sutnick citizens' forum",
    r"office of education and performance initiatives",
    r"recruitment, testing and audit division",
    r"training & development division",
    r"title ix of the education amendments",
    r"meet fearless community leaders who serve on our board",
    r"reserve your spot now for the 2026 arena scholastic",
    r"all official records.*recommendation forms.*translated into english",
    r"students are expected to perform well academically.*admission assessment",
    r"showing consistent, above-average achievement.*a's and b's",
    r"we offer limited extra support options to students in fourth",
    r"student success network on-campus tutoring",
    r"admission.*portal.*apply now.*button",
    r"carrollton admission portal account",
    r"it is our goal through this student-centered process",
    r"the mission of the office of college advising",
    r"our people are the key to our past success",
    r"administration, students, faculty, parents, alumni",
    r"palmer trinity is a place to build a career",
    r"faculty turnover rate.*below 10%.*national average",
]

def is_junk(text: str) -> bool:
    if not text:
        return True
    tl = text.lower().strip()
    return any(re.search(p, tl) for p in JUNK_PATTERNS)


# ── IMAGE HARVESTING ──────────────────────────────────────────────────────────

SKIP_IMG = [
    "logo", "icon", "svg", "1x1", "pixel", "spinner", "placeholder",
    "blank", "transparent", "avatar", "flag", "arrow", "bullet",
    "check", "star", "rating", "rs/facebook", "rs/linkedin", "rs/youtube",
    "loupe", "panier", "menu", "profil", "default_image", "black-beacon",
    "weather/128",
    # Miami-specific UI noise
    "m80-book-camp", "img-golf-club-2022-award", "imga-world-cup-banner",
    "bradesco_bank", "sports-illustrated",
    "vanderbilt", "dartmouth", "brown", "unc-chapel",  # university logos
    "shapes-1-1", "vertical-ipad",
]
EXTS_IMG = [".jpg", ".jpeg", ".png", ".webp"]

async def harvest_images(pg, base_url: str, image_set: set):
    try:
        await pg.evaluate("""async () => {
            await new Promise(resolve => {
                let total = 0;
                const dist = 400;
                const timer = setInterval(() => {
                    window.scrollBy(0, dist);
                    total += dist;
                    if (total >= document.body.scrollHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }
                }, 80);
            });
        }""")
        await asyncio.sleep(1)
    except Exception:
        pass

    try:
        for img in await pg.query_selector_all("img"):
            for attr in ["src", "data-src", "data-lazy-src", "data-original",
                         "data-image", "data-bg", "data-lazy", "data-srcset"]:
                c = await img.get_attribute(attr)
                if c:
                    src = c.strip().split(" ")[0].split(",")[0].strip()
                    full = urljoin(base_url, src)
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)
                            and len(full) > 25):
                        image_set.add(full)
            srcset = await img.get_attribute("srcset")
            if srcset:
                parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    full = urljoin(base_url, parts[-1])
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)):
                        image_set.add(full)

        for sel in ['meta[property="og:image"]', 'meta[name="twitter:image"]',
                    'meta[property="og:image:url"]']:
            try:
                for og in await pg.evaluate(
                    f"() => Array.from(document.querySelectorAll('{sel}')).map(m=>m.content)"
                ):
                    if og and og.strip() and not any(s in og.lower() for s in SKIP_IMG):
                        image_set.add(og.strip())
            except Exception:
                pass

        bg_urls = await pg.evaluate("""() => {
            const hits = [];
            const sels = [
                '[style*="background-image"]','[class*="hero"]','[class*="banner"]',
                '[class*="slider"]','[class*="slide"]','[class*="bg-image"]',
                '[class*="cover"]','section','header'
            ];
            sels.forEach(sel => {
                try {
                    document.querySelectorAll(sel).forEach(el => {
                        const s = el.style.backgroundImage ||
                                  getComputedStyle(el).backgroundImage || '';
                        const m = s.match(/url\\(["']?([^"')]+)["']?\\)/);
                        if (m && m[1]) hits.push(m[1]);
                    });
                } catch(e) {}
            });
            return [...new Set(hits)];
        }""")
        for bg in bg_urls:
            if bg:
                full = urljoin(base_url, bg.strip())
                if (any(e in full.lower() for e in EXTS_IMG)
                        and not any(s in full.lower() for s in SKIP_IMG)):
                    image_set.add(full)
    except Exception:
        pass


# ── QUOTE SCRAPER ─────────────────────────────────────────────────────────────

def _word_count(text):
    return len(text.split())

def scrape_quote_from_soup(soup):
    SKIP_PHRASES = [
        "cookie", "menu", "nav", "©", "privacy", "terms",
        "subscribe", "sign up", "click", "read more", "javascript",
        "watch the video", "watch video", "listen to", "testimonial",
        "share", "follow us", "contact us", "apply now", "find out more",
        "learn more", "get in touch", "download", "register", "book a",
        "visit us", "dialog window", "escape will cancel",
        "site you are about to visit", "no longer exists",
        # Miami-specific quote junk
        "reserve your spot now for the 2026",
        "dog parks, fitness",
        "enter camper details",
        "we surveyed 150 institutions",
        "gerald w. moore",
        "i am a child of the sacred heart",
        "education is the solution to every problem",
    ]
    selectors = [
        "blockquote", "q",
        "[class*='quote']", "[class*='pullquote']", "[class*='callout']",
        "[class*='hero'] p", "[class*='banner'] p",
        "[class*='mission'] p", "[class*='vision'] p",
        "figcaption",
    ]
    candidates = []
    for sel in selectors:
        for el in soup.select(sel):
            text = re.sub(r'\s+', ' ', el.get_text()).strip()
            text = text.strip('\u201c\u201d\u2018\u2019"\'')
            wc   = _word_count(text)
            if wc < 8 or wc > 35:
                continue
            if any(x in text.lower() for x in SKIP_PHRASES):
                continue
            if is_junk(text):
                continue
            if not re.search(r'[.!?]$', text):
                continue
            candidates.append(text)
    if not candidates:
        return ""
    candidates.sort(key=lambda t: _word_count(t))
    return candidates[0]


# ── AI QUOTE ──────────────────────────────────────────────────────────────────

def generate_ai_quote(school_name, school_type, city, about_text):
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return f"Excellence, integrity, and curiosity — the pillars of {school_name}."
    prompt = (
        f"Write ONE short inspirational quote (1 sentence, 10–20 words) "
        f"that reflects the philosophy of {school_name}, a {school_type} in {city}. "
        f"Context: {about_text[:300]}. "
        f"Sound like the organisation's mission statement. "
        f"Return ONLY the quote text — no quotation marks, no attribution, no explanation."
    )
    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 60, "temperature": 0.7,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data  = json.loads(resp.read())
            quote = data["choices"][0]["message"]["content"].strip().strip('"\'')
            words = quote.split()
            if len(words) > 25:
                quote = " ".join(words[:25])
            print(f"  ✓ AI quote generated for {school_name}")
            return quote
    except Exception as e:
        print(f"  ⚠ AI quote failed: {e}")
    return f"Excellence, integrity, and curiosity — the pillars of {school_name}."


# ── OPENAI FORCE-FILL ALL PERFORMANCE FIELDS ──────────────────────────────────

def openai_force_fill_performance(results: dict) -> dict:
    """
    Force-fills ALL 6 performance fields.
    Any field blank, N/A, too short (<12 words), or junk gets
    a unique, substantive 2-sentence AI response.
    NEVER overwrites a genuinely good scraped value.
    """
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    name        = results.get("Name", "this organisation")
    entity_type = results.get("Type", "Independent School")
    city        = results.get("City", "Miami, FL")
    about       = results.get("About", "")
    philosophy  = results.get("Philosophy", "")

    ctx_parts = [f"Name: {name}", f"Type: {entity_type}", f"City: {city}"]
    if about:
        ctx_parts.append(f"About: {about}")
    if philosophy:
        ctx_parts.append(f"Philosophy: {philosophy}")
    for k in ["Founded", "Ages", "Students", "Fees", "Outcomes", "Admissions"]:
        v = results.get(k, "")
        if v and v not in ("N/A", "See website", ""):
            ctx_parts.append(f"{k}: {v}")
    context = "\n".join(ctx_parts)

    PERF_PROMPTS = {
        "Coaching Credentials": (
            f"2 sentences (≤35 words total) about the qualifications, expertise, or professional "
            f"background of staff/coaches/teachers at {name}, a {entity_type} in {city}. "
            f"Mention certifications, experience level, or teaching specialisms. Be specific."
        ),
        "Student Wellbeing": (
            f"2 sentences (≤35 words total) about how {name} supports the mental, emotional, "
            f"or physical wellbeing of students/members. Mention counselling, advisory, "
            f"community building, or pastoral programmes. Be specific."
        ),
        "Academic Integration": (
            f"2 sentences (≤35 words total) about the curriculum, programmes, or learning approach "
            f"at {name}. Mention specific courses, qualifications, or teaching methods. Be specific."
        ),
        "Competitive Pathway": (
            f"2 sentences (≤35 words total) about the results, achievements, university "
            f"destinations, or career pathways of graduates from {name}. "
            f"Mention college admissions outcomes or competition results. Be specific."
        ),
        "Facilities & Resources": (
            f"2 sentences (≤35 words total) about the physical facilities, campus spaces, "
            f"or resources at {name}. "
            f"Mention specific amenities — studios, labs, courts, pools, fields, gyms. Be specific."
        ),
        "Ongoing Accountability": (
            f"2 sentences (≤35 words total) about how {name} tracks student/athlete progress, "
            f"provides feedback, or monitors quality. "
            f"Mention progress reports, parent communication, or assessment systems. Be specific."
        ),
    }

    # Extended junk list for Miami-specific scraping noise
    PERF_JUNK_PHRASES = [
        "cookie", "privacy policy", "gdpr", "javascript",
        "img academy will provide sheets", "pillow and blanket",
        "camper did not receive sheets", "your suggestion box",
        "community satisfaction survey",
        "document.getelementbyid", "ak_js",
        "accessibility statement", "skip to main content",
        "city commission", "city manager", "commissioner",
        "officers of the board", "remaining trustees",
        "it is both a privilege and a sacred duty",
        "heather is an alumna of carrollton",
        "we warmly welcome you to the 2025",
        "parents are responsible for requesting",
        "these must be submitted electronically",
        "rental.*facilities.*based on availability",
        "please connect with your service specialist",
        "number of campers per room",
        "esl course with weekly themes",
        "inspiring people.*extreme sports",
        "custom-curated experience.*movies poolside",
        "©.*tpr education", "princeton review is not affiliated",
        "sat is a registered trademark",
        "if your starting score is below 1250",
        "we surveyed 150 institutions",
        "best classroom experience.*best college radio station",
        "study when and where it works best.*by the pool",
        "dog parks.*fitness.*sports.*playgrounds",
        "times for the public comment period",
        "office of education and performance initiatives",
        "recruitment, testing and audit division",
        "training & development division",
        "title ix of the education amendments",
        "meet fearless community leaders who serve on our board",
        "reserve your spot now for the 2026 arena scholastic",
        "all official records.*recommendation forms",
        "students are expected to perform well academically.*admission assessment",
        "we offer limited extra support options",
        "student success network on-campus tutoring",
        "admission.*portal.*apply now.*button",
        "carrollton admission portal",
        "mission of the office of college advising",
        "our people are the key to our past success",
        "administration, students, faculty, parents, alumni",
        "palmer trinity is a place to build a career",
        "faculty turnover rate.*below 10%",
        "need to create an equal opportunity base for polo",
        "it was officially launched on january 20, 1967",
        "the need to create an equal opportunity base",
    ]

    def needs_fill(val: str) -> bool:
        if not val or val.strip() in ("N/A", "", "N/a", "See website"):
            return True
        clean = re.sub(r'\s+', ' ', val).strip()
        if len(clean.split()) < 12:
            return True
        cl = clean.lower()
        return any(p in cl for p in PERF_JUNK_PHRASES) or is_junk(clean)

    missing = {
        metric: prompt
        for metric, prompt in PERF_PROMPTS.items()
        if needs_fill(results.get("Performance", {}).get(metric, ""))
    }

    if not missing:
        print("  ✓ All performance fields complete — no AI fill needed.")
        return results

    print(f"  → AI force-filling {len(missing)} performance field(s): {', '.join(missing.keys())}")

    field_lines = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are an expert on Miami and South Florida educational institutions, sports academies, arts organisations, and wellness programmes.

KNOWN DATA about this institution:
{context}

Generate UNIQUE, FACTUAL, SPECIFIC content for ONLY these fields.
Each field must:
- Be exactly 2 sentences, ≤35 words total
- Describe a DIFFERENT aspect from the other fields
- Sound professional and informative (not generic)
- Use your knowledge of {name} in {city} to be accurate
- Never repeat facts from another field

Reply ONLY with valid JSON, keys exactly as shown below. No markdown, no extra keys:

{field_lines}

Return format: {{"field name": "2 sentence content here.", ...}}"""

    payload = json.dumps({
        "model": "gpt-4o",
        "max_tokens": 1200,
        "temperature": 0.4,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=40) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            seen_vals   = set()
            for metric, val in filled.items():
                if not val or val.strip() in ("N/A", ""):
                    continue
                key_short = re.sub(r'\s+', ' ', val.strip().lower())[:100]
                if key_short in seen_vals:
                    continue
                seen_vals.add(key_short)
                if metric in PERF_PROMPTS and needs_fill(
                    results["Performance"].get(metric, "")
                ):
                    results["Performance"][metric] = str(val).strip()
                    filled_keys.append(metric)

            if filled_keys:
                print(f"  ✓ AI filled performance: {', '.join(filled_keys)}")
            else:
                print("  ✓ AI performance: nothing new added.")

    except json.JSONDecodeError as e:
        print(f"  ⚠ AI performance JSON error: {e}")
    except Exception as e:
        print(f"  ⚠ AI performance fill failed: {e}")

    return results


# ── OPENAI MAIN FIELDS GAP-FILL ───────────────────────────────────────────────

def openai_fill_missing_fields(results: dict) -> dict:
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    entity_type = results.get("Type", "Independent School")
    is_school   = any(w in entity_type.lower()
                      for w in ["school", "academy", "college", "education",
                                "programme", "program", "arts"])
    name = results.get("Name", "this organisation")

    FILL_FIELDS = {
        "Founded":    "Year this organisation was founded (just the 4-digit year). If unknown reply N/A.",
        "Ages":       ("Student age range (e.g. '4–18'). If unknown reply N/A." if is_school
                       else "Age range of members/clients (e.g. 'All ages', 'Adults 18+', '5–18')."),
        "Students":   ("Approximate enrolled students. If unknown reply N/A." if is_school
                       else "Approximate annual members/participants/visitors. If unknown reply N/A."),
        "Ratio":      "Staff-to-student or instructor-to-client ratio (e.g. '1:8'). If unknown reply N/A.",
        "Fees":       "Annual tuition or membership in USD. Miami private school range typically $25,000–$55,000/year. If unknown reply 'See website'.",
        "About":      f"2 sentences (30–45 words) describing {name}'s core identity, mission, and what makes it distinctive.",
        "Philosophy": f"2 sentences (30–45 words) on {name}'s specific teaching, coaching, or organisational approach.",
        "Outcomes":   f"2 sentences (30–45 words) on concrete outcomes from {name} — university placements, awards, competition results.",
        "Admissions": f"2 sentences (30–45 words) on how to apply or enrol at {name} — process, timeline, requirements.",
        "Tagline":    f"One compelling tagline (≤20 words) for {name}.",
    }

    ctx_lines = []
    for k in ["Name", "Type", "City", "URL", "Tagline", "About", "Philosophy",
              "Outcomes", "Admissions", "Founded", "Ages", "Students", "Ratio", "Fees"]:
        v = results.get(k, "")
        if v:
            ctx_lines.append(f"{k}: {v}")
    context = "\n".join(ctx_lines) or f"URL: {results.get('URL', '')}"

    def _needs(val):
        return not val or str(val).strip() in ("N/A", "See website", "Rolling admissions",
                                                "Year-round", "", "N/a")

    def _needs_para(field, val):
        if _needs(val):
            return True
        if field in ("About", "Philosophy", "Outcomes", "Admissions"):
            if len(str(val).split()) < 20:
                return True
            return is_junk(str(val))
        return False

    missing = {}
    for f, p in FILL_FIELDS.items():
        v = results.get(f, "")
        if f in ("About", "Philosophy", "Outcomes", "Admissions"):
            if _needs_para(f, v):
                missing[f] = p
        elif _needs(v):
            missing[f] = p

    if not missing:
        return results

    print(f"  → OpenAI filling {len(missing)} main field(s): {', '.join(missing.keys())}")

    field_instructions = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are a Miami and South Florida education, arts, and sports expert.
Use the known data AND your knowledge to fill missing fields accurately for {name}.

KNOWN DATA:
{context}

Fill ONLY these fields (reply with a JSON object, keys exactly as shown):
{field_instructions}

Rules:
- Use factual knowledge of this specific Miami / South Florida institution.
- Fees: use $ and per year. Miami private school range typically $25,000–$55,000/year.
- Ages: always provide a range, even for non-schools (e.g. 'All ages', 'Adults 18+', '4–18').
- Paragraph fields (About, Philosophy, Outcomes, Admissions): 2 full sentences, 30–45 words total. Never write fewer than 25 words.
- Each paragraph must be specific and substantive — not generic filler.
- Do NOT add commentary, markdown fences, or extra keys.
- Reply with valid JSON only."""

    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 1800, "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            for fkey, val in filled.items():
                if not val or str(val).strip() in ("N/A", "See website", ""):
                    continue
                if fkey in FILL_FIELDS:
                    v = results.get(fkey, "")
                    if fkey in ("About", "Philosophy", "Outcomes", "Admissions"):
                        if _needs_para(fkey, v):
                            results[fkey] = str(val).strip()
                            filled_keys.append(fkey)
                    elif _needs(v):
                        results[fkey] = str(val).strip()
                        filled_keys.append(fkey)

            if filled_keys:
                print(f"  ✓ OpenAI filled: {', '.join(filled_keys)}")
            else:
                print("  ✓ OpenAI: nothing new to fill.")
    except Exception as e:
        print(f"  ⚠ OpenAI main fill failed: {e}")

    return results


# ── WP BLOCKS GENERATOR (6 images + PERF_SHORT keywords) ─────────────────────

def generate_wp_blocks(r):
    school_name  = r.get("Name", "")
    description  = r.get("Tagline", "")
    age          = r.get("Ages", "")
    students     = r.get("Students", "")
    ratio        = r.get("Ratio", "")
    founded      = r.get("Founded", "")
    school_type  = r.get("Type", "")
    city         = r.get("City", "")
    annual_fee   = r.get("Fees", "")
    about        = r.get("About", "")
    philosophy   = r.get("Philosophy", "")
    outcomes     = r.get("Outcomes", "")
    quote        = r.get("Quote", "")
    website_url  = r.get("URL", "")
    admissions   = r.get("Admissions", "")
    images       = r.get("Images", [])
    perf         = r.get("Performance", {})
    enquiry_open = r.get("EnquiryOpen", "Year-round")
    app_deadline = r.get("AppDeadline", "Rolling admissions")

    website_display = website_url.replace("https://", "").replace("http://", "").rstrip("/")
    city_slug  = city.lower().replace(", ", "-").replace(" ", "-").replace(",", "")
    city_short = city.split(",")[0].strip()

    # Always 6 image slots — pad with img0 if needed
    imgs = [img for img in images if img]
    while len(imgs) < 6:
        imgs.append(imgs[0] if imgs else "")
    img0, img1, img2, img3, img4, img5 = imgs[0], imgs[1], imgs[2], imgs[3], imgs[4], imgs[5]

    coaching       = perf.get("Coaching Credentials", "")
    wellbeing      = perf.get("Student Wellbeing", "")
    academic       = perf.get("Academic Integration", "")
    competitive    = perf.get("Competitive Pathway", "")
    facilities     = perf.get("Facilities & Resources", "")
    accountability = perf.get("Ongoing Accountability", "")

    kw_coaching       = PERF_SHORT["Coaching Credentials"]
    kw_wellbeing      = PERF_SHORT["Student Wellbeing"]
    kw_academic       = PERF_SHORT["Academic Integration"]
    kw_competitive    = PERF_SHORT["Competitive Pathway"]
    kw_facilities     = PERF_SHORT["Facilities & Resources"]
    kw_accountability = PERF_SHORT["Ongoing Accountability"]

    wp = f"""<!-- wp:image {{"id":440,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="{school_name}" class="wp-image-440"/></figure>
<!-- /wp:image -->

<!-- wp:paragraph -->
<p><a href="elite-home.html">Elite</a> &#8250; <a href="elite-city-{city_slug}.html">{city_short}</a> &#8250; <a href="elite-program-academic.html">Elite Academic Programs</a></p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Kidrovia Elite — Listed</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Elite Academic Programs</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":1}} -->
<h1 class="wp-block-heading"><em>{school_name}</em></h1>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{description}</p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{age}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{students}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{ratio}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{founded}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column {{"width":"66.66%"}} -->
<div class="wp-block-column" style="flex-basis:66.66%">
<!-- wp:heading --><h2 class="wp-block-heading">About <em>{school_name}</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{about}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column {{"width":"33.33%"}} -->
<div class="wp-block-column" style="flex-basis:33.33%">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Institution Details</h5><!-- /wp:heading -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Type</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">City</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{school_type}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{age}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{students}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{ratio}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{founded}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{city}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{annual_fee}</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Contact &amp; Enquiry</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading"><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display} &#8594;</a></h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">Quote</h5>
<!-- /wp:heading -->
<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Philosophy</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">How they <em>teach</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{philosophy}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Outcomes</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Where students <em>go</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{outcomes}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img1}" alt="{school_name} campus"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img2}" alt="{school_name} students"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img3}" alt="{school_name} activities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img4}" alt="{school_name} facilities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img5}" alt="{school_name} community"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":4}} -->
<h4 class="wp-block-heading">Our Assessment</h4>
<!-- /wp:heading -->
<!-- wp:heading -->
<h2 class="wp-block-heading"><strong>How {school_name} performs</strong></h2>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_coaching}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Coaching Credentials</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{coaching}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_wellbeing}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Student Wellbeing</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{wellbeing}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_academic}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Academic Integration</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{academic}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_competitive}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Competitive Pathway</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{competitive}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_facilities}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Facilities &amp; Resources</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{facilities}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_accountability}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Ongoing Accountability</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{accountability}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading --><h2 class="wp-block-heading">Admissions &amp;<br><em>How to Apply</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{admissions}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Enquiries Open</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{enquiry_open}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Application Deadline</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{app_deadline}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{annual_fee}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Location</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{city}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Website</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display}</a></p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->
"""
    return wp


# ── MAIN SCRAPER ──────────────────────────────────────────────────────────────

async def extract_school_data(url):
    async with async_playwright() as p:
        import platform
        headless = os.environ.get(
            "HEADLESS",
            "true" if platform.system() == "Linux" else "false"
        ).lower() == "true"
        browser = await p.chromium.launch(headless=headless, slow_mo=0 if headless else 200)
        context = await browser.new_context(viewport={"width": 1280, "height": 800})
        page    = await context.new_page()

        results = {
            "Name": "", "Tagline": "", "Founded": "", "City": "",
            "Ages": "", "Students": "", "Ratio": "", "Fees": "",
            "Type": "Independent School",
            "About": "", "Philosophy": "", "Outcomes": "",
            "Admissions": "", "Quote": "",
            "EnquiryOpen": "Year-round",
            "AppDeadline": "Rolling admissions",
            "Performance": {}, "URL": url, "Images": [],
        }

        global_text_pool = []

        # ── STEP 0: Lock name, city, and known data from domain map ──────────
        domain = url.split("/")[2].replace("www.", "")
        known_entry = next(
            ((k, v) for k, v in KNOWN_NAMES_MIA.items() if k in domain or k in url),
            None
        )
        known_domain_key = None
        if known_entry:
            k_fragment, (k_name, k_city) = known_entry
            results["Name"] = k_name
            results["City"] = k_city   # CITY LOCK — never overridden by body text
            known_domain_key = next(
                (dk for dk in KNOWN_DATA_MIA if dk in domain or dk in url), None
            )
            if known_domain_key:
                for field, val in KNOWN_DATA_MIA[known_domain_key].items():
                    results[field] = val
            print(f"  ✓ Known institution: {k_name} | City locked: {k_city}")

        FALLBACKS = {
            "Founded": "", "Ages": "", "Students": "", "Ratio": "N/A", "Fees": "See website",
        }

        fee_patterns = [
            r'\$\s*([\d,]+)(?:\.\d{2})?\s*per\s*(?:year|annum|semester|month)',
            r'tuition\s*(?:fee)?s?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
            r'annual\s*(?:tuition|fee)s?\s*(?:of|is|:)?\s*\$\s*([\d,]+)',
            r'fee[s]?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
        ]

        def extract_fee_validated(text: str) -> str:
            for pat in fee_patterns:
                for m in re.finditer(pat, text, re.I):
                    try:
                        num = int(m.group(1).replace(',', ''))
                        if 1000 <= num <= 100000:
                            return m.group(0).strip()
                    except (IndexError, ValueError):
                        pass
            return ""

        ratio_patterns = [
            r'(\d+\s*:\s*\d+)\s*(?:student[- ]to[- ](?:faculty|teacher)|teacher[- ]to[- ]student|ratio)',
            r'ratio\s*(?:of\s*)?(\d+\s*:\s*\d+)',
            r'(\d+\s*:\s*\d+)\s*ratio',
            r'1\s*(?:teacher|member of staff|coach|instructor)\s+(?:to|per|:)\s+(\d+)\s*(?:students?|pupils?|learners?)',
            r'(\d+)\s*(?:students?|pupils?|learners?)\s+(?:per|to)\s+(?:every\s+)?(?:1\s+)?(?:teacher|tutor|coach)',
        ]

        age_patterns = [
            r'students?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'children\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'from\s+age\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'(\d+)\s*(?:to|[\u2013\-])\s*(\d+)\s*years?\s*(?:old)',
            r'grades?\s+(\d+)\s*(?:through|to|[\u2013\-])\s*(\d+)',
            r'grade\s+(\d+)\s*(?:through|to|[\u2013\-])\s*(?:grade\s+)?(\d+)',
        ]

        student_patterns = [
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:pupils?|students?|boys|girls|young people|young women|learners?|scholars?)',
            r'(?:approximately|around|about|some|over|more than)\s+(\d[\d,]+)\s*(?:pupils?|students?|girls?|learners?)',
            r'school\s+of\s+(?:approximately|around|about)?\s*(\d[\d,]+)',
            r'enroll(?:ment|ment|ing)\s+(?:of\s+)?(?:approximately|about)?\s*(\d[\d,]+)',
            r'community\s+of\s+(?:over\s+)?(\d[\d,]+)\s*(?:students?|learners?|members?)',
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:student|pupil|learner|child|member)',
        ]

        print(f"Connecting to: {url}...")
        loaded = await smart_goto(page, url)
        if not loaded:
            print(f"  ✗ Could not load {url}, skipping.")
            await browser.close()
            return None

        try:
            await handle_cookies_automatically(page)

            og_site   = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:site_name\"]'); return m ? m.content : ''; }")
            og_title  = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:title\"]'); return m ? m.content : ''; }")
            raw_title = await page.title()

            REJECT_NAMES   = {
                "home", "welcome", "homepage", "index", "main", "", "welcome to",
                "co", "test prep tutoring", "about", "no title", "(no title)",
            }
            REJECT_STARTS  = ("welcome to ", "home -", "home |", "top private school")
            REJECT_PHRASES = [
                "the polo training foundation is dedicated",
                "athletic training & performance camps at",
                "personalization", "advertising effectiveness",
            ]

            def clean_name(candidate: str) -> str:
                parts = re.split(r'\s*[|—]\s*', candidate)
                parts = [p.strip() for p in parts if p.strip()]
                GENERIC = {"home","welcome","index","main","page","site","portal","homepage","co","about",""}
                n = parts[0] if parts else candidate.strip()
                if n.lower() in GENERIC and len(parts) > 1:
                    n = parts[-1].strip()
                for prefix in ("welcome to ",):
                    if n.lower().startswith(prefix):
                        n = n[len(prefix):].strip()
                return n

            # Only set name if not already locked from KNOWN_NAMES_MIA
            if not results["Name"]:
                for candidate in [og_site, og_title, raw_title]:
                    name = clean_name(candidate)
                    if (name
                            and name.lower() not in REJECT_NAMES
                            and not any(name.lower().startswith(p) for p in REJECT_STARTS)
                            and not any(bad in name.lower() for bad in REJECT_PHRASES)
                            and len(name.split()) <= 8):
                        results["Name"] = name
                        break
                if not results["Name"]:
                    domain_part = url.split("/")[2].replace("www.", "")
                    results["Name"] = domain_part.split(".")[0].replace("-", " ").title()

            body_text = await page.inner_text("body")

            if len(body_text.strip()) < 300:
                print("  ⚠ Page body too short — scrolling to trigger JS render…")
                try:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(3)
                    await page.evaluate("window.scrollTo(0, 0)")
                    await asyncio.sleep(2)
                    body_text = await page.inner_text("body")
                except Exception:
                    pass

            # Only set city from body if not already locked
            if not results["City"]:
                results["City"] = _detect_city_from_body(body_text, url)

            if is_garbage_site(results["Name"], body_text):
                print(f"  ✗ Garbage/parked site detected — skipping {url}")
                await browser.close()
                return None

            def extract(pattern):
                m = re.search(pattern, body_text, re.IGNORECASE)
                return m.group(1).strip() if m else ""

            current_year = datetime.now().year

            def extract_founded_from_text(text, existing_candidates=None):
                STRONG_KWS  = ["founded in", "established in", "was founded", "founding year",
                                "founded by", "foundress", "charter", "incorporated in",
                                "opened in", "opened its doors", "since", "est."]
                WEAK_KWS    = ["founded", "established", "foundation", "history", "began"]
                SKIP_WORDS  = ["copyright", "©", "reserved", "updated", "class of",
                                "alumni", "graduate", "batch of"]
                FOREIGN_CTX = ["in the uk", "in england", "parent school", "sister school"]
                cands = existing_candidates if existing_candidates is not None else []
                clean = re.sub(r'\s+', ' ', text.lower())
                clean = re.sub(r'(copyright|©).*?\d{4}', '', clean)
                for sent in re.split(r'[.!?\n]', clean):
                    if any(w in sent for w in SKIP_WORDS):
                        continue
                    has_strong = any(k in sent for k in STRONG_KWS)
                    has_weak   = any(k in sent for k in WEAK_KWS)
                    if not (has_strong or has_weak):
                        continue
                    is_foreign = any(ctx in sent for ctx in FOREIGN_CTX)
                    for y in re.findall(r'\b(1[6-9]\d{2}|20[012]\d)\b', sent):
                        year = int(y)
                        if not (1600 <= year <= current_year - 1):
                            continue
                        score = 3 if has_strong else 1
                        if year < 1900:
                            score -= 2
                        if is_foreign:
                            score -= 3
                        if year >= current_year - 2:
                            score -= 6
                        cands.append((score, year))
                return cands

            # Only scrape Founded if not locked
            if not results.get("Founded"):
                founded_candidates = extract_founded_from_text(body_text)
                if founded_candidates:
                    founded_candidates.sort(key=lambda x: (-x[0], x[1]))
                    results["Founded"] = str(founded_candidates[0][1])

            # Only scrape Ages if not locked
            if not results.get("Ages"):
                age_candidates = []
                for pat in age_patterns:
                    for m in re.finditer(pat, body_text, re.I):
                        try:
                            g = m.groups()
                            if len(g) >= 2 and g[0] and g[1]:
                                lo, hi = int(g[0]), int(g[1])
                                if lo < hi and lo >= 2 and hi <= 25:
                                    age_candidates.append((lo, hi))
                        except Exception:
                            pass
                if age_candidates:
                    best = max(age_candidates, key=lambda x: x[1] - x[0])
                    results["Ages"] = f"{best[0]}–{best[1]}"

                if not results["Ages"]:
                    AGE_MAP = {
                        "prek": (4, 5), "pre-k": (4, 5), "pre-kindergarten": (4, 5),
                        "preschool": (3, 5), "nursery": (3, 4),
                        "kindergarten": (5, 6), "kinder": (5, 6),
                        "grade 1": (6, 7), "1st grade": (6, 7),
                        "grade 6": (11, 12), "6th grade": (11, 12),
                        "grade 9": (14, 15), "9th grade": (14, 15),
                        "grade 12": (17, 18), "12th grade": (17, 18),
                        "elementary school": (5, 11), "lower school": (5, 11),
                        "middle school": (11, 14), "upper school": (14, 18),
                        "high school": (14, 18),
                        "college preparatory": (14, 18), "college prep": (14, 18),
                    }
                    tl = body_text.lower()
                    fy = [v for k, v in AGE_MAP.items() if k in tl]
                    results["Ages"] = f"{min(x[0] for x in fy)}–{max(x[1] for x in fy)}" if fy else ""

            # Only scrape Students if not locked
            if not results.get("Students"):
                for pat in student_patterns:
                    for m in re.finditer(pat, body_text, re.I):
                        num_str = re.sub(r'[\s,]', '', m.group(1))
                        try:
                            num = int(num_str)
                            if not (50 <= num <= 10000):
                                continue
                            if 2020 <= num <= 2030:
                                continue
                            ctx = body_text[max(0, m.start()-8):m.end()+8]
                            if '%' in ctx or '.' in m.group(1):
                                continue
                            results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                            break
                        except ValueError:
                            pass
                    if results["Students"]:
                        break

            if not results.get("Ratio"):
                for pat in ratio_patterns:
                    m = re.search(pat, body_text, re.I)
                    if m:
                        results["Ratio"] = m.group(1).replace(" ", "")
                        break

            results["Fees"] = results.get("Fees") or extract_fee_validated(body_text)

            # Type detection — only if not locked from KNOWN_DATA_MIA
            if not results.get("Type") or results["Type"] == "Independent School":
                tl    = body_text.lower()
                url_l = url.lower()
                if "imgacademy.com" in url_l:
                    results["Type"] = "Sports Performance Academy & Boarding School"
                elif "polotraining.org" in url_l or ("polo" in tl and "foundation" in tl):
                    results["Type"] = "Polo Training Foundation"
                elif "princetonreview.com" in url_l or "kaplan.com" in url_l:
                    results["Type"] = "Academic Test Prep & Tutoring"
                elif "miamimusicproject.org" in url_l:
                    results["Type"] = "Youth Music Programme"
                elif "nwsa.mdc.edu" in url_l or "new world school" in tl:
                    results["Type"] = "Performing & Visual Arts School"
                elif "baptist" in url_l and "yoga" in url_l:
                    results["Type"] = "Yoga Training & Education Platform"
                elif "crandon" in url_l and "tennis" in url_l:
                    results["Type"] = "Public Tennis Centre"
                elif "miamibeach" in url_l and "tennis" in url_l:
                    results["Type"] = "Public Tennis Centre"
                elif "all-girls" in tl or "girls' school" in tl or "girls school" in tl:
                    results["Type"] = "Independent All-Girls School"
                elif "all-boys" in tl or "boys' school" in tl:
                    results["Type"] = "Independent Boys' School"
                elif "montessori" in tl:
                    results["Type"] = "Montessori School"
                elif "ib world" in tl or "international baccalaureate" in tl:
                    results["Type"] = "IB World Day School"
                elif "boarding" in tl and "day" in tl:
                    results["Type"] = "Independent Boarding & Day School"
                elif "boarding" in tl:
                    results["Type"] = "Independent Boarding School"
                elif "college prep" in tl or "college preparatory" in tl:
                    results["Type"] = "College Preparatory School"
                elif "charter school" in tl:
                    results["Type"] = "Charter School"
                elif "magnet school" in tl:
                    results["Type"] = "Magnet School"
                elif "tennis" in tl and ("academy" in tl or "center" in tl or "centre" in tl):
                    results["Type"] = "Tennis Academy"
                elif "polo" in tl and ("club" in tl or "training" in tl):
                    results["Type"] = "Polo Club & Training"
                elif "yoga" in tl and ("studio" in tl or "center" in tl):
                    results["Type"] = "Yoga Training & Education Platform"
                elif "music" in tl and ("programme" in tl or "program" in tl or "project" in tl):
                    results["Type"] = "Youth Music Programme"
                elif "performing arts" in tl or "visual arts" in tl:
                    results["Type"] = "Performing & Visual Arts School"
                elif "sports" in tl and ("academy" in tl or "camp" in tl or "training" in tl):
                    results["Type"] = "Sports Performance Academy"
                elif "test prep" in tl or "sat prep" in tl:
                    results["Type"] = "Academic Test Prep & Tutoring"
                elif "elementary" in tl and "high school" in tl:
                    results["Type"] = "Independent K-12 School"
                elif "high school" in tl or "upper school" in tl:
                    results["Type"] = "Independent High School"
                elif "middle school" in tl:
                    results["Type"] = "Independent Middle School"
                elif "elementary" in tl or "lower school" in tl:
                    results["Type"] = "Independent Elementary School"
                else:
                    results["Type"] = "Independent School"

            soup_home = BeautifulSoup(await page.content(), "html.parser")
            for attr in [{"name": "description"}, {"property": "og:description"}]:
                tag = soup_home.find("meta", attr)
                if tag and tag.get("content"):
                    content = tag["content"][:200]
                    if not is_junk(content):
                        results["Tagline"] = content
                        break
            if not results["Tagline"]:
                candidate = extract(r"([A-Z][^.]{30,150}\.)")
                if candidate and not is_junk(candidate):
                    results["Tagline"] = candidate

            scraped_quote = scrape_quote_from_soup(soup_home)
            if scraped_quote and not is_junk(scraped_quote):
                results["Quote"] = scraped_quote

            image_set = set()
            await harvest_images(page, url, image_set)
            results["Images"] = [img for img in image_set if img][:10]

        except Exception as e:
            print(f"Home page error: {e}")
            await browser.close()
            return None

        # ── SUBPAGE CRAWL ─────────────────────────────────────────────────────
        soup  = BeautifulSoup(await page.content(), "html.parser")
        links = soup.find_all("a", href=True)
        queue     = []
        seen_urls = {url.rstrip("/")}

        targets = {
            "about": ["about", "history", "welcome", "mission", "who-we-are",
                      "our-story", "overview", "vision", "values", "ethos"],
            "outcomes": ["results", "destination", "university", "leavers", "beyond",
                         "college", "achievements", "academic-results", "college-counseling",
                         "college-placement", "matriculation"],
            "fees": ["fees", "tuition", "financial", "cost", "charges",
                     "fee-structure", "school-fees", "annual-fees", "pricing",
                     "affording", "financial-aid"],
            "admission": ["admissions", "apply", "entry", "register", "enroll",
                          "how-to-apply", "application", "join-us", "registration",
                          "enrolment", "admission-process", "open-house"],
            "facilities": ["facilities", "campus", "co-curriculum", "sport", "arts",
                           "library", "infrastructure", "resources", "activities",
                           "co-curricular", "extracurricular", "programs", "academics"],
        }

        for link in links:
            href, text = link["href"], link.get_text().lower().strip()
            full_url   = urljoin(url, href).split("#")[0].rstrip("/")
            if full_url not in seen_urls and url.split("/")[2] in full_url:
                if any(k in text or k in href.lower() for cat in targets.values() for k in cat):
                    queue.append((full_url, text))
                    seen_urls.add(full_url)

        used_sentences: set = set()

        def _register(text: str):
            if not text:
                return
            for s in re.split(r'(?<=[.!?])\s+', text.strip()):
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and len(key) > 20:
                    used_sentences.add(key)

        def dedup_text(text: str) -> str:
            if not text:
                return text
            parts = re.split(r'(?<=[.!?])\s+', text.strip())
            fresh = []
            for s in parts:
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and key not in used_sentences and not is_junk(s):
                    fresh.append(s.strip())
                    used_sentences.add(key)
            return " ".join(fresh)

        def assign_field(field_name: str, text: str, max_sent: int = 2):
            if results.get(field_name) and not is_junk(results[field_name]):
                return
            cleaned = dedup_text(clean_to_sentences(text, max_sent))
            if cleaned and not is_junk(cleaned):
                results[field_name] = cleaned

        for _field in ["About", "Tagline", "Philosophy", "Outcomes", "Admissions", "Quote"]:
            _register(results.get(_field, ""))

        used_paras_global: set = set()

        # Strict per-metric keyword lists (non-overlapping)
        PERF_KWS = {
            "Coaching Credentials":   [
                "qualified", "certified", "credential", "faculty", "expertise",
                "experienced teacher", "master", "phd", "trained", "professional development",
                "instructor background", "staff background", "degree in",
            ],
            "Student Wellbeing":      [
                "wellbeing", "mental health", "counselor", "counselling", "advisory",
                "pastoral", "social-emotional", "support program", "community",
                "belonging", "college counselor", "wellness program",
            ],
            "Academic Integration":   [
                "curriculum", "coursework", "program of study", "academic program",
                "core curriculum", "learning outcomes", "course offerings",
                "academic requirements", "subject", "ap course", "honors",
                "ib diploma", "advanced placement",
            ],
            "Competitive Pathway":    [
                "university placement", "college acceptance", "accepted to",
                "graduates attend", "alumni", "college counseling outcome",
                "ivy league", "top university", "competition winner",
                "national merit", "scholarship", "award winner", "matriculation",
            ],
            "Facilities & Resources": [
                "gymnasium", "theater", "theatre", "laboratory", "library",
                "swimming pool", "pool", "studio", "auditorium", "makerspace",
                "science lab", "art studio", "music room", "dance studio",
                "athletic field", "sports facility", "60 acres", "campus spans",
            ],
            "Ongoing Accountability": [
                "progress report", "parent conference", "student assessment",
                "feedback system", "portfolio", "transcript", "grade report",
                "parent communication", "accreditation", "nais",
                "annual review", "learning plan", "parent portal",
            ],
        }

        PERF_SKIP = [
            "img academy will provide sheets", "pillow and blanket",
            "camper did not receive", "your suggestion box",
            "community satisfaction survey",
            "officers of the board", "remaining trustees",
            "it is both a privilege and a sacred duty",
            "heather is an alumna of carrollton",
            "we warmly welcome you to the 2025",
            "parents are responsible for requesting",
            "these must be submitted electronically",
            "rental.*facilities.*based on availability",
            "please connect with your service specialist",
            "number of campers per room",
            "esl course with weekly themes",
            "custom-curated experience.*movies poolside",
            "©.*tpr education", "sat is a registered trademark",
            "if your starting score is below",
            "we surveyed 150 institutions",
            "dog parks.*fitness.*sports.*playgrounds",
            "times for the public comment period",
            "office of education and performance initiatives",
            "recruitment, testing and audit division",
            "training & development division",
            "title ix of the education amendments",
            "meet fearless community leaders who serve on our board",
            "mission of the office of college advising",
            "our people are the key to our past success",
            "palmer trinity is a place to build a career",
            "need to create an equal opportunity base for polo",
            "it was officially launched on january 20, 1967",
            "carrollton does not discriminate in admissions",
        ]

        for target_url, link_text in queue[:15]:
            try:
                await page.goto(target_url, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)
                main          = page.locator("main").first
                content_scope = main if await main.is_visible() else page
                paragraphs    = await content_scope.locator("p").all_text_contents()
                clean_paras   = [
                    t.strip() for t in paragraphs
                    if len(t.strip()) >= 60
                    and not any(x in t.lower() for x in ["cookie", "privacy", "menu", "navigation", "footer"])
                    and not is_junk(t)
                ]
                global_text_pool.extend(clean_paras)
                full_text = " ".join(clean_paras)
                page_text = await page.inner_text("body")
                sub_soup  = BeautifulSoup(await page.content(), "html.parser")

                await harvest_images(page, url, image_set)
                results["Images"] = [img for img in image_set if img][:10]

                if any(k in link_text or k in target_url for k in targets["about"]):
                    clean_about = [p for p in clean_paras
                                   if not is_junk(p)
                                   and not p.lower().startswith((
                                       "please fill", "please note", "click here",
                                       "the admissions team", "fill the", "select",
                                       "it is both a privilege",
                                   ))]
                    assign_field("About", " ".join(clean_about), 2)
                    teach_kws  = ["curriculum", "teaching", "learning", "approach",
                                  "education", "ethos", "method", "ib", "philosophy",
                                  "program", "pedagogy", "inquiry", "project-based"]
                    teach_para = [p for p in clean_paras
                                  if any(k in p.lower() for k in teach_kws)
                                  and not is_junk(p)]
                    assign_field("Philosophy", " ".join(teach_para), 2)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q
                    if not results.get("Founded"):
                        sub_cands = extract_founded_from_text(page_text)
                        if sub_cands:
                            sub_cands.sort(key=lambda x: (-x[0], x[1]))
                            results["Founded"] = str(sub_cands[0][1])

                if not results.get("Students"):
                    for pat in student_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            num_str = m.group(1).replace(",", "")
                            try:
                                num = int(num_str)
                                if 50 <= num <= 10000 and not (2020 <= num <= 2030):
                                    results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                    break
                            except ValueError:
                                pass

                if not results.get("Ratio"):
                    for pat in ratio_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            results["Ratio"] = m.group(1).replace(" ", "")
                            break

                if any(k in link_text or k in target_url for k in targets["outcomes"]):
                    outcome_kws = ["college", "university", "accepted", "acceptance",
                                   "placement", "graduate", "alumni", "destination",
                                   "enrolled at", "attend", "scholarship", "matriculate"]
                    outcome_paras = [p for p in clean_paras
                                     if any(k in p.lower() for k in outcome_kws)
                                     and not is_junk(p)]
                    assign_field("Outcomes", " ".join(outcome_paras) or full_text, 2)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q

                if any(k in link_text or k in target_url for k in targets["fees"]):
                    fee_val = extract_fee_validated(page_text)
                    if fee_val and not results.get("Fees"):
                        results["Fees"] = fee_val

                if any(k in link_text or k in target_url for k in targets["admission"]):
                    adm_kws  = ["apply", "application", "admission", "register",
                                "assessment", "entry", "enroll", "enrolment",
                                "registration", "open house", "inquiry", "shadow day"]
                    filtered = [p for p in clean_paras
                                if any(k in p.lower() for k in adm_kws)
                                and not is_junk(p)]
                    assign_field("Admissions", " ".join(filtered), 2)
                    dl = re.search(
                        r'(?:deadline|closes?|closing date)\s*[:\-]?\s*([^\n.]{10,60})',
                        page_text, re.I
                    )
                    if dl and results["AppDeadline"].startswith("Rolling"):
                        val = dl.group(1).strip()[:80]
                        has_month = re.search(
                            r'\b(january|february|march|april|may|june|july|august|'
                            r'september|october|november|december|jan|feb|mar|apr|'
                            r'jun|jul|aug|sep|oct|nov|dec)\b', val, re.I)
                        has_date = re.search(r'\b\d{1,2}[\/\-]\d{1,2}', val)
                        if (has_month or has_date) and not is_junk(val):
                            results["AppDeadline"] = val
                    if not results.get("Ages"):
                        for pat in age_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                try:
                                    g = m.groups()
                                    if len(g) >= 2 and g[0] and g[1]:
                                        lo, hi = int(g[0]), int(g[1])
                                        if lo < hi and lo >= 2 and hi <= 25:
                                            results["Ages"] = f"{lo}–{hi}"
                                except Exception:
                                    pass
                                if results["Ages"]:
                                    break

                if any(k in link_text or k in target_url for k in targets["facilities"]):
                    for metric, kws in PERF_KWS.items():
                        if not results["Performance"].get(metric):
                            matched = [
                                p for p in clean_paras
                                if any(k in p.lower() for k in kws)
                                and p not in used_paras_global
                                and not is_junk(p)
                                and not any(sk in p.lower() for sk in PERF_SKIP)
                                and len(p.split()) >= 15
                            ]
                            if matched:
                                chosen = matched[:1]
                                val    = dedup_text(clean_to_sentences(" ".join(chosen), 2))
                                if val and len(val.split()) >= 12 and not is_junk(val):
                                    results["Performance"][metric] = val
                                    used_paras_global.update(chosen)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q

            except Exception as e:
                print(f"  Skipping {target_url}: {e}")
                continue

        results["Images"] = list(image_set)[:10]
        for field, fallback in FALLBACKS.items():
            if not results.get(field):
                results[field] = fallback
        if not results["City"]:
            results["City"] = "Miami, FL"

        if not results["Quote"]:
            print(f"  No quote found — generating AI quote…")
            results["Quote"] = generate_ai_quote(
                results["Name"], results["Type"], results["City"], results["About"]
            )

        await browser.close()

        # ── STEP 1: Fill main fields ──────────────────────────────────────────
        print(f"\n  Running OpenAI main field gap-fill…")
        results = openai_fill_missing_fields(results)

        # ── STEP 2: Force-fill ALL 6 performance fields (never blank/N/A/junk)
        print(f"\n  Running AI force-fill for performance fields…")
        results = openai_force_fill_performance(results)

        # ── POST SANITIZATION ─────────────────────────────────────────────────
        for field in ["Name", "Tagline", "Founded", "City", "Ages", "Students",
                      "Ratio", "Fees", "Type", "About", "Philosophy", "Outcomes",
                      "Admissions", "Quote"]:
            if results.get(field):
                results[field] = re.sub(r'\s+', ' ', str(results[field])).strip()
        for field in ["About", "Philosophy", "Outcomes", "Admissions"]:
            if results.get(field) and len(results[field].split()) > 60:
                results[field] = clean_to_sentences(results[field], 2)
        PERF_METRICS = ["Coaching Credentials", "Student Wellbeing", "Academic Integration",
                        "Competitive Pathway", "Facilities & Resources", "Ongoing Accountability"]
        for metric in PERF_METRICS:
            val = results["Performance"].get(metric, "")
            if val and len(val.split()) > 50:
                results["Performance"][metric] = clean_to_sentences(val, 2)

        # ── PRINT SUMMARY ─────────────────────────────────────────────────────
        SEP        = "—" * 55
        city_short = results["City"].split(",")[0]
        print(f"\n{SEP}")
        print(f"Elite › {city_short} › {results['Name']}")
        print(SEP)
        print(f"\n  Type       : {results['Type']}")
        print(f"  Ages       : {results['Ages'] or 'N/A'}")
        print(f"  Students   : {results['Students'] or 'N/A'}")
        print(f"  Ratio      : {results['Ratio'] or 'N/A'}")
        print(f"  Founded    : {results['Founded'] or 'N/A'}")
        print(f"  City       : {results['City']}")
        print(f"  Annual Fee : {results['Fees']}")
        print(f"  Images     : {len(results['Images'])} found")
        for i, img in enumerate(results["Images"], 1):
            print(f"    {i}. {img}")
        print(f"\n  About      : {results['About']}")
        print(f"\n  Quote      : \"{results['Quote']}\"")
        print(f"\n  Philosophy : {results['Philosophy']}")
        print(f"\n  Outcomes   : {results['Outcomes']}")
        print(f"\n  Admissions : {results['Admissions']}")
        print(f"\nPerformance Metrics:")
        for k, v in results["Performance"].items():
            kw = PERF_SHORT.get(k, k)
            print(f"\n  [{kw}] {k}:")
            print(f"  {v or '❌ STILL MISSING'}")
        print(f"\n  Website : {results['URL']}")
        print(SEP)

        # ── SAVE ──────────────────────────────────────────────────────────────
        wp_code     = generate_wp_blocks(results)
        school_slug = re.sub(r"[^a-z0-9]+", "_", results["Name"].lower())[:40].strip("_")
        out_wp      = f"{school_slug}_wp_blocks.txt"
        out_json    = f"{school_slug}_data.json"

        with open(out_wp, "w", encoding="utf-8") as f:
            f.write(wp_code)

        save_data               = {k: v for k, v in results.items() if k != "Images"}
        save_data["ImageCount"] = len(results.get("Images", []))
        save_data["ImageURLs"]  = results.get("Images", [])
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)

        print(f"\n  WP blocks  → {out_wp}")
        print(f"  JSON data  → {out_json}")
        return results


# ── CITY FALLBACK (body-text only, out-of-region cities blocked) ──────────────

MIAMI_BODY_HINTS = {
    "coral gables":     "Coral Gables, FL",
    "coconut grove":    "Coconut Grove, FL",
    "miami beach":      "Miami Beach, FL",
    "miami gardens":    "Miami Gardens, FL",
    "palmetto bay":     "Palmetto Bay, FL",
    "pinecrest":        "Pinecrest, FL",
    "key biscayne":     "Key Biscayne, FL",
    "homestead":        "Homestead, FL",
    "aventura":         "Aventura, FL",
    "brickell":         "Miami, FL",
    "wynwood":          "Miami, FL",
    "kendall":          "Miami, FL",
    "hialeah":          "Hialeah, FL",
    "doral":            "Doral, FL",
    "boca raton":       "Boca Raton, FL",
    "fort lauderdale":  "Fort Lauderdale, FL",
    "west palm beach":  "West Palm Beach, FL",
    "wellington":       "Wellington, FL",
    "palm beach":       "Palm Beach, FL",
    "bradenton":        "Bradenton, FL",
    "sarasota":         "Sarasota, FL",
    "orlando":          "Orlando, FL",
    "tampa":            "Tampa, FL",
    "miami":            "Miami, FL",
    "florida":          "Miami, FL",
}

def _detect_city_from_body(body_text: str, url: str) -> str:
    url_lower = url.lower()
    for hint in sorted(MIAMI_BODY_HINTS, key=len, reverse=True):
        if hint in url_lower:
            return MIAMI_BODY_HINTS[hint]
    sample = body_text[:5000].lower()
    for hint in sorted(MIAMI_BODY_HINTS, key=len, reverse=True):
        if hint in sample:
            return MIAMI_BODY_HINTS[hint]
    return "Miami, FL"


# ── URLS ──────────────────────────────────────────────────────────────────────

school_urls = [
    "https://ransomeverglades.org",
    "https://gulliverschools.org",
    "https://carrollton.org",
    "https://palmertrinity.org",
    "https://miamicountryday.org",
    "https://www.miamibeachfl.gov/city-hall/parks-and-recreation/parks-facilities-directory/miami-beach-tennis-center/",
    "https://polotraining.org/",
    "https://crandonparktennis.com/",
    "https://princetonreview.com",
    "https://www.imgacademy.com/sport-camps/performance-camp",
    "https://baptisteyoga.com",
    "https://miamimusicproject.org",
    "https://nwsa.mdc.edu/",
]


async def run_multiple_schools():
    all_results = []
    for url in school_urls:
        print(f"\n{'=' * 60}")
        print(f"  Scraping: {url}")
        print(f"{'=' * 60}")
        try:
            result = await extract_school_data(url)
            if result:
                all_results.append(result)
            await asyncio.sleep(3)
        except Exception as e:
            print(f"  Failed: {url} → {e}")

    master_file = "all_schools_data_miami.json"
    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n{'=' * 60}")
    print(f"All done! {len(all_results)} institutions scraped.")
    print(f"Master JSON → {master_file}")
    print(f"{'=' * 60}")
    return all_results


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future   = pool.submit(asyncio.run, run_multiple_schools())
            all_data = future.result()
    except RuntimeError:
        asyncio.run(run_multiple_schools())


  Scraping: https://ransomeverglades.org
  ✓ Known institution: Ransom Everglades School | City locked: Coconut Grove, FL
Connecting to: https://ransomeverglades.org...
  ⚠ networkidle failed (TimeoutError), retrying…
  No quote found — generating AI quote…
  ✓ AI quote generated for Ransom Everglades School

  Running OpenAI main field gap-fill…
  → OpenAI filling 4 main field(s): About, Philosophy, Outcomes, Admissions
  ✓ OpenAI filled: About, Philosophy, Outcomes, Admissions

  Running AI force-fill for performance fields…
  → AI force-filling 6 performance field(s): Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability
  ✓ AI filled performance: Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability

———————————————————————————————————————————————————————
Elite › Coconut Grove › Ransom Everglades School
——————————————————————————